# ANÁLISE DE CUSTOMERFEEDBACK (Terra Signal)

Este notebook interpreta o texto de CustomerFeedback (em inglês), classifica o sentimento por cliente e aspecto e gera as tabelas de apoio ao atendimento. Escopo: Churn e CustomerFeedback.

## Dados
- Catálogo: hackaton_databricks, esquema tabelas_hackaton.
- Origem: history_tipada (5.634 clientes, com Churn) e inference_tipada (1.409 clientes, sem Churn).
- Terra Signal (colega): arquivos CSV carregados em tabela (aspects, feedback_aspects e intelligence).
- Tabelas geradas: feedback_frases, feedback_aspectos, feedback_sentimento e feedback_consolidado (history e inference); feedback_classificacao_history; terra_signal_history_intelligence_Adri e terra_signal_customer_intelligence_Adri.

## Meta
- Acerto mínimo de 75% sobre todos os clientes (Churn Yes e No), comparando a classificação do texto com a coluna Churn da history.
- A data do feedback é desconhecida. O texto pode divergir do Churn atual, e isso conta como erro.

## Etapas
1. Frases: uma linha por frase.
2. Aspectos: uma linha por frase e aspecto citado.
3. Sentimento: de cada citação (frase x aspecto), pela frase inteira.
4. Consolidação: uma linha por cliente e aspecto.
5. Classificação por cliente (R1, R2 e R) e acerto por regra (só history, pois a inference não tem Churn).
6. Sentimento global e VADER por cliente, aspecto principal e orientação de atendimento.
7. Comparação com a Terra Signal (history e inference).

As etapas 1 a 4 se repetem na history e na inference.

## Formato da consolidação
- Colunas: customerID, Aspecto, Sentimento_Consolidado, Qtd_Mencoes, Qtd_Positivas, Qtd_Negativas, Qtd_Mistas, Evidencia_Mista, Evidencia_Negativa, Evidencia_Positiva.
- Uma linha por cliente e aspecto. Aspecto não citado não gera linha.
- Sentimento_Consolidado: Positivo, Negativo, Misto ou Sem avaliacao.
- Qtd_Mencoes: frases que citam o aspecto, incluindo as sem avaliação.
- Qtd_Mistas: 0 ou 1. A citação mista não entra em Positivas nem Negativas.
- Evidências: frase inteira; várias frases unidas por " | ".

## Decisões de método
1. A frase termina em ".", "!" ou "?" seguido de espaço. Valores monetários não são cortados.
2. Aspectos do modelo: Cobranca, Conexao, Contrato, Preco. Acrescentados: Atendimento, Streaming, Seguranca e Saida. Telefone, Backup e Protecao não tiveram citações.
3. Atendimento é o atendimento ao cliente, não o serviço TechSupport.
4. Saida: o cliente informa que saiu (Negativo) ou que não sai (Positivo), sem citar aspecto específico. Exige termo de saída na frase (churn, cancel, switch, discontinu, leav).
5. Limitação: a frase inteira vale para todos os aspectos que ela cita. A divisão em fragmentos (vírgula, "but", "however") não foi feita.
6. Streaming: parte das citações é uso da internet ("stream movies"), não o serviço contratado.
7. Símbolo de moeda seguido de número não pode ser escrito no código (o Databricks lê como parâmetro). Usa-se \x24 na regex.
8. Dicionário de sentimento ampliado com fair, works well, flexibility, accommodates (positivos) e better options, preferred more, competitive pricing (negativos).
9. Todo cliente da history tem aspecto Saida, então o SentimentoGlobal nunca é Sem avaliacao na history. O feedback não tem texto vazio, então SentimentScore e SentimentoVADER não têm nulos.

## Dados da base
- history: 5.634 clientes, 25.362 frases (4,5 por cliente), 26,7% com Churn Yes.
- inference: 1.409 clientes, sem Churn.
- Citações por aspecto na history: Conexao 8.708, Preco 6.938, Cobranca 6.228, Saida 5.048, Contrato 2.420, Atendimento 171, Streaming 87, Seguranca 1.

## Pontos em aberto
- Streaming e Seguranca têm volume baixo e ficam fora da análise dos motivos.
- Contrato citado no texto (month-to-month, one-year, two-year) x Churn: consulta não rodada.

**VALIDAÇÕES**

**1. Frases sem avaliação + saída: churn de 91,6% a 100%, entrada e saída iguais**
- Questionado: as frases de Conexao e Contrato sem avaliação que citam saída falam de saída ou de entrada ("switched to", "signed up")?
- Resultado: o churn ficou entre 91,6% e 100% nos dois tipos. A separação entre Entrada e Saida não distinguiu os clientes.
- Decisão: separação Entrada/Saida descartada. Essas frases têm termo de saída e o cliente citou um aspecto, então o aspecto é a causa da saída.

**2. Acerto texto x Churn: R1 100%, R 97% e R2 87%**
- Questionado: o texto indica quem saiu? Meta: acerto mínimo de 75% sobre todos os clientes (Churn Yes e No). O valor foi arbitrário, sem expectativa de atingi-lo.
- Resultado:
  - R1: 1 falso positivo e 0 falsos negativos.
  - R2: 168 falsos positivos e 565 falsos negativos.
  - R: 169 falsos positivos e 0 falsos negativos.
- Decisão: o problema estava em Saida. Antes, toda frase com "churn" era classificada como saída, tanto para clientes Yes quanto No. A inclusão da inversão ("no churn", "no plans to churn" = não saída) elevou o acerto acima da meta. R1, R2 e R atingem a meta.

**3. Contagem por sentimento consolidado (history): Positivo 66,5% e Sem avaliacao 8,9%**
- Questionado: como o consolidado distribui os sentimentos por cliente e aspecto?
- Resultado: Positivo 66,5% (15.482 linhas), Misto 14,2% (3.308), Negativo 10,5% (2.436), Sem avaliacao 8,9% (2.063).

**4. Concordância das regras com o Churn (history): R1 100%, R 97%, R2 87%**
- Questionado: quanto R1, R2 e R acertam, e onde erram?
- Resultado:
  - R1: 1 falso positivo e 0 falsos negativos. Precisão 99,9% e cobertura 100%.
  - R2: 168 falsos positivos e 565 falsos negativos. Precisão 84,8% e cobertura 62,4%.
  - R: 169 falsos positivos e 0 falsos negativos. Precisão 89,9% e cobertura 100%.
- Decisão: as três regras atingem a meta de 75%. A R1 é a que mais acerta, e a R2 é a que mais deixa passar clientes com Churn Yes.

**5. Contagem por sentimento consolidado (inference): Positivo 65,9% e Sem avaliacao 8,9%**
- Questionado: como o consolidado distribui os sentimentos por cliente e aspecto na inference?
- Resultado: Positivo 65,9% (3.855 linhas), Misto 14,9% (872), Negativo 10,3% (603), Sem avaliacao 8,9% (522).

**6. Contagem por sentimento na Terra Signal (history): Sem avaliacao 49,2% e Positivo 47,2%**
- Questionado: como a Terra Signal distribui os sentimentos por cliente e aspecto?
- Resultado: Sem avaliacao 49,2% (9.384 linhas), Positivo 47,2% (9.006), Negativo 2,7% (515), Misto 0,9% (169).
- Decisão: o consolidado avalia mais e capta mais negativos e mistos que a Terra Signal, influencia da melhoria no regex conforme Validação 2.

**7. Contagem por sentimento na Terra Signal (inference): Sem avaliacao 48,8%**
- Questionado: o padrão se repete na inference?
- Resultado: Sem avaliacao 48,8%, Negativo 2,7%.
- Decisão: mesma conclusão da validação 6.

**8. Clientes do consolidado x Terra Signal (history): mesmos 5.634 clientes**
- Questionado: as duas tabelas têm os mesmos clientes?
- Resultado: 5.634 clientes nas duas, sem clientes exclusivos de uma delas.
- Decisão: a comparação por cliente e aspecto é válida.

**9. Sentimento do consolidado x Terra Signal (history): a Terra Signal avalia menos**
- Questionado: o sentimento do consolidado é o mesmo na Terra Signal, para o mesmo cliente e aspecto?
- Resultado:
  - 15.482 Positivo no consolidado = 7.252 Positivo + 5.637 Sem avaliacao + 2 Negativo + 1 Misto + 2.590 ausentes na Terra Signal
  - 2.436 Negativo no consolidado = 263 Negativo + 597 Sem avaliacao + 11 Positivo + 4 Misto + 1.561 ausentes na Terra Signal
  - 3.308 Misto no consolidado = 164 Misto + 1.743 Positivo + 250 Negativo + 1.124 Sem avaliacao + 27 ausentes na Terra Signal
  - 2.063 Sem avaliacao no consolidado = 2.025 Sem avaliacao + 38 ausentes na Terra Signal
- Decisão: o consolidado avalia mais e capta mais negativos e mistos. A Terra Signal deixa sem avaliação o que o consolidado classifica.

**10. Aspectos do consolidado x Terra Signal (history): Saida, Seguranca e Streaming só no consolidado**
- Questionado: os aspectos são os mesmos nas duas tabelas?
- Resultado (linhas no consolidado x linhas na Terra Signal):
  - Atendimento: 160 x 160
  - Cobranca: 5.599 x 5.402
  - Conexao: 5.621 x 5.619
  - Contrato: 2.370 x 2.369
  - Preco: 5.610 x 5.523
  - Saida: 3.841 x ausente
  - Seguranca: 1 x ausente
  - Streaming: 87 x ausente
  - Suporte: ausente x 1
- Decisão: manter Saida, Seguranca e Streaming. A Terra Signal não capta Saida: quando o cliente diz que saiu, vai sair ou está saindo (por exemplo, usando "churn"), sem elogio ou crítica específica, não há sentimento aparente, mas é Negativo por ser cancelamento ou positivo por afirmação de pretenção a cancelamento.

**11. Aspectos do consolidado x Terra Signal (inference): Saida e Streaming só no consolidado**
- Questionado: os aspectos são os mesmos nas duas tabelas?
- Resultado (linhas no consolidado x linhas na Terra Signal):
  - Atendimento: 32 x 32
  - Cobranca: 1.393 x 1.339
  - Conexao: 1.407 x 1.407
  - Contrato: 610 x 610
  - Preco: 1.400 x 1.381
  - Saida: 986 x ausente
  - Streaming: 24 x ausente
- Decisão: mesma conclusão da Validação 10. A Terra Signal não capta Saida, cliente que diz que saiu, vai sair ou está saindo, sem elogio ou crítica específica, que é Negativo por ser cancelamento e em contrapartida informa que não tem pretensão de sair, o que significa que está no mínimo satisfeito com o serviço, embora não de forma explicita com algum aspecto . Streaming também não aparece nela.

**12. Linhas por sentimento, consolidado x Terra Signal (history): Sem avaliacao 2.063 x 9.384**
- Questionado: quantas linhas cada tabela tem por sentimento?
- Resultado (consolidado x Terra Signal, diferença = consolidado menos Terra Signal):
  - Misto: 3.308 x 169 (+3.139)
  - Negativo: 2.436 x 515 (+1.921)
  - Positivo: 15.482 x 9.006 (+6.476)
  - Sem avaliacao: 2.063 x 9.384 (−7.321)
- Decisão: reflexo da melhoria do código em si, mais termos para avaliação, e da inclusão do sentimento de saída (ou de permanência).E com Saida, o acerto contra o Churn sobe (Validação 4: R1 100%, R 97%, R2 87%).

**13. Limites do SentimentoVADER no colega: Neutro de −0,0387 a 0,0498**
- Questionado: o limite de ±0,05 vale para o SentimentoVADER da tabela original com classificação de sentimentos sem Saída, para replicação
- Resultado: Neutro de −0,0387 a 0,0498, Positivo a partir de 0,0516, Negativo até −0,0511. O VADER calculado por nós coincidiu com o score do colega (diferença 0,0).
- Decisão: SentimentScore é o compound do VADER (4 casas). SentimentoVADER é Positivo (>= 0,05), Negativo (<= -0,05) ou Neutro.

**Análises anteriores (células removidas)**
- Saida Negativo em Churn No (634 clientes): a maioria é "fica" que a regex atual não captura ("no intentions", "no reason to churn", "not considered switching"). Uma regex nova marcaria 552 de 638 como "fica", mas não foi aplicada: a atual já passa da meta (R1: 88,7%).
- Dicionário ampliado (fair, works well, flexibility, accommodates, better options, preferred more, competitive pricing): Sem avaliacao dos quatro aspectos principais passou de 3.757 para 3.561 citações. Aplicado.
- Uma melhoria possível no regex poderia ser ao invés de quebrar em frases inteiras, quebrar em frações de frases, como marcadores como "," ou expressões como "but", mas seria necessário estudo e testes para validação. 

**COMO O CLIENTE É CLASSIFICADO (CustomerFeedback)**

Cada CustomerFeedback é dividido em frases. Cada frase é ligada aos aspectos que cita. Cada citação (frase + aspecto) recebe um sentimento. As citações do cliente decidem se ele é classificado como churn Yes ou No.

**1. Aspectos: o que conta como citação**
A frase conta como citação do aspecto quando contém um destes termos (em qualquer forma, sem distinguir maiúsculas). Se contém termos de mais de um aspecto, conta em cada um.
- Conexao: internet, dsl, fiber, speed(s), connection, connectivity, outage(s), downtime, disruption(s), interruption(s)
- Preco: símbolo de moeda seguido de número, charge(s), price(s), pricing, expensive, afford..., cost(s), steep, deal(s), cheap..., fee(s)
- Cobranca: pay, pays, paying, paid, payment(s), check(s), bank transfer(s), credit card, bill(s), billing, paperless
- Contrato: contract(s), month-to-month, one-year, two-year, one year, two year
- Atendimento: tech support, technical support, customer service, customer support, support
- Streaming: stream..., movie(s), tv
- Seguranca: security
- Saida: churn..., cancel..., switch..., discontinu..., leav...
- Telefone (phone, multiple lines), Backup e Protecao: sem citações em history.
Aspecto não citado não gera linha.

**2. Sentimento de aspecto (todos, exceto o aspecto Saida, tratado no item 3)**
Nesta ordem:
1. Elogio negado vira queixa: negação (not, never, no longer, n't) + até 2 palavras + convenient, reliable, fast, consistent, reasonable, satisfied, happy, good, affordable, ideal, worth, recommend..., meet ou resolved. Ex.: "not convenient".
2. Queixa negada vira elogio: no, without, never experienced, haven't had + issues, problems, complaints, interruptions, outages, downtime, disruptions, hassle, delays ou disappointments. Ex.: "no issues with downtime".
3. Contagem das palavras que sobram:
 - Positivas: reasonable, reliable, reliably, fast, convenient, conveniently, satisfied, satisfactory, happy, great, good, affordable, appreciate..., pleased, excellent, seamless, hassle-free, responsive, helpful, worth, recommend..., positive, easy, consistent, consistently, top-notch, impressed, enjoyed, smooth, stable, fair, works well, flexib..., accommodat...
 - Negativas: high, higher, expensive, steep, slow, slower, unreliable, inconvenient, inconvenience, outdated, disappointed, disappointing, disappointment, issue(s), problem(s), frustrat..., cumbersome, hassle, lacking, lack, poor, unexpected, outage(s), interruption(s), disruption(s), inconsistent, unresolved, dissatisfied, unsatisfied, drawback, too, difficult, delay(s), unhappy, better deal, better pricing, better options, preferred more, competitive pricing
4. Resultado: só positivas = Positivo. Só negativas = Negativo. As duas = Misto. Nenhuma = Sem avaliacao.
O sentimento vale para a frase inteira, então vale para todos os aspectos que ela cita.

**3. Sentimento do aspecto Saida**
- Saida Positivo ("fica"): frase com negação (no, not, never, n't), até 3 palavras depois plan(s), intention, intend... ou "issue(s) with", e até 3 palavras depois churn, cancel, switch ou leav. Ex.: "no plans to churn".
- Saida Negativo: frase do aspecto Saida sem esse padrão. O código não procura "eu saí" de forma explícita.
- Saida não tem Sem avaliacao nem Misto.

**4. Como se chega a churn Yes ou No**
- Para cada cliente, a regra olha as citações dele. Basta existir uma citação com sentimento Negativo, no aspecto definido pela regra (ver item 5), para classificar como churn Yes. Não há soma nem peso: uma única citação decide, mesmo que o resto do texto seja elogio.
- Sem citação do tipo, ou sem nenhuma citação, classifica como churn No.
- A classificação do texto é comparada com a coluna Churn do history. Iguais = acerto. Diferentes = erro. Acerto geral = acertos ÷ 5.634 clientes.
- Falso positivo: texto classifica Yes, coluna Churn = No. Falso negativo: texto classifica No, coluna Churn = Yes.
Nota: Misto é considerado Churn No porque não há como mostrar a inclinação (positiva ou negativa).

**5. Regras**
- R1 - Sentimento do aspecto Saida: se existe pelo menos uma citação com sentimento Negativo, Churn Yes. Considera o que o cliente diz sobre sair e não indica motivo.
- R2 - Sentimento dos aspectos Conexao, Preco, Cobranca e Contrato: se existe pelo menos uma citação Negativo, Churn Yes.
- R - Sentimento de todos os aspectos (R1 + R2): se existe pelo menos uma citação Negativo em Saida, Conexao, Preco, Cobranca ou Contrato, Churn Yes. Sem citação Negativo, Churn No.

In [0]:
%pip install vaderSentiment
# ============================================================
# Instalação da biblioteca VADER
# ============================================================
# O que faz: instala o pacote vaderSentiment, usado para calcular o SentimentScore e o SentimentoVADER.
# Origem:    repositório de pacotes Python do cluster
# Saída:     biblioteca vaderSentiment (versão 3.3.2) disponível no notebook.
# Regras:    deve rodar antes das células de sentimento por cliente (history e inference). Pode exigir reiniciar o Python (dbutils.library.restartPython()).
# ============================================================

Looking in indexes: [REDACTED]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%python
dbutils.library.restartPython()

# ============================================================
# Reinício do Python após a instalação do VADER
# ============================================================
# O que faz: reinicia o Python do notebook para o pacote vaderSentiment ser reconhecido.
# Origem:    célula anterior (%pip install vaderSentiment)
# Saída:     ambiente Python reiniciado, com o vaderSentiment disponível.
# Regras:    deve rodar logo após a instalação e antes de qualquer célula de código. Variáveis criadas antes dela são perdidas.
# ============================================================

In [0]:
%sql
-- ============================================================
-- Frases do feedback (history)
-- ============================================================
-- O que faz: divide o CustomerFeedback em frases, uma linha por frase, mantendo customerID e Churn. É a base das células de aspectos e sentimento.
-- Origem:    history_tipada
-- Saída:     tabela feedback_frases_history (customerID, Churn, id_frase, frase).
-- Regras:    a frase termina em ".", "!" ou "?" seguido de espaço ou quebra de linha, então valores como 73.9 não são cortados. Frases vazias e feedbacks nulos são descartados. id_frase começa em 0. A regex usa "\\s" (barra dupla), exigida em SQL no Databricks.
-- ============================================================


CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_frases_history AS
SELECT
  h.customerID,
  h.Churn,
  p.pos AS id_frase,
  trim(p.col) AS frase
FROM hackaton_databricks.tabelas_hackaton.history_tipada h,
     LATERAL posexplode(split(h.CustomerFeedback, '(?<=[.!?])\\s+')) AS p
WHERE h.CustomerFeedback IS NOT NULL
  AND trim(p.col) <> '';

SELECT
  COUNT(DISTINCT customerID) AS clientes,
  COUNT(*) AS frases,
  ROUND(COUNT(*) / COUNT(DISTINCT customerID), 1) AS frases_por_cliente
FROM hackaton_databricks.tabelas_hackaton.feedback_frases_history;

clientes,frases,frases_por_cliente
5634,25362,4.5


In [0]:
%sql
-- ============================================================
-- Citações por frase e aspecto (history)
-- ============================================================
-- O que faz: detecta, por frase, quais aspectos são citados, usando um dicionário de termos por aspecto. Só marca a menção; o sentimento vem na célula seguinte.
-- Origem:    feedback_frases_history
-- Saída:     tabela feedback_aspectos_history (customerID, Churn, id_frase, frase, Aspecto), uma linha por frase e aspecto.
-- Regras:    o dicionário é em regex, sem distinção de maiúsculas e minúsculas, e uma frase pode gerar mais de um aspecto. Aspectos: Conexao, Preco, Cobranca, Contrato, Telefone, Seguranca, Backup, Protecao, Atendimento, Streaming e Saida. Os termos vêm da leitura de 100 textos e da lista de serviços. Preco inclui símbolo de moeda seguido de número, escrito como código hexadecimal x24 para o Databricks não lê-lo como parâmetro. Streaming inclui "stream", "movie" e "tv", e parte das menções descreve uso da internet, não o serviço contratado. Saida marca só a menção (churn, cancel, switch, discontinu, leav), sem tratar negação; a negação é tratada na célula de sentimento.

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_aspectos_history AS
WITH dicionario AS (
  SELECT * FROM VALUES
    ('Conexao',   r'(?i)\b(internet|dsl|fiber|speeds?|connection|connectivity|outages?|downtime|disruptions?|interruptions?)\b'),
    ('Preco',     r'(?i)(\x24\s?\d|\b(charges?|prices?|pricing|expensive|afford\w*|costs?|steep|deals?|cheap\w*|fees?)\b)'),
    ('Cobranca',  r'(?i)\b(pay|pays|paying|paid|payments?|checks?|bank transfers?|credit card|bill|bills|billing|paperless)\b'),
    ('Contrato',  r'(?i)\b(contracts?|month-to-month|one-year|two-year|one year|two year)\b'),
    ('Telefone',  r'(?i)\b(phone|multiple lines?)\b'),
    ('Seguranca', r'(?i)\bsecurity\b'),
    ('Backup',    r'(?i)\bback-?up\b'),
    ('Protecao',  r'(?i)\bprotection\b'),
    ('Atendimento',   r'(?i)\b(tech support|technical support|customer service|customer support|support)\b'),
    ('Streaming', r'(?i)\b(stream\w*|movies?|tv)\b'),
    ('Saida',     r'(?i)\b(churn\w*|cancel\w*|switch\w*|discontinu\w*|leav\w+)\b')
  AS d(Aspecto, regex)
)
SELECT
  f.customerID,
  f.Churn,
  f.id_frase,
  f.frase,
  d.Aspecto
FROM hackaton_databricks.tabelas_hackaton.feedback_frases_history f
JOIN dicionario d
  ON f.frase rlike d.regex;

SELECT
  Aspecto,
  COUNT(*) AS frases,
  COUNT(DISTINCT customerID) AS clientes,
  COUNT(DISTINCT CASE WHEN Churn = 'Yes' THEN customerID END) AS clientes_churn,
  ROUND(100 * COUNT(DISTINCT CASE WHEN Churn = 'Yes' THEN customerID END) / COUNT(DISTINCT customerID), 1) AS pct_churn
FROM hackaton_databricks.tabelas_hackaton.feedback_aspectos_history
GROUP BY Aspecto
ORDER BY frases DESC;

Aspecto,frases,clientes,clientes_churn,pct_churn
Conexao,8708,5621,1502,26.7
Preco,6938,5610,1498,26.7
Cobranca,6228,5599,1469,26.2
Saida,5048,3841,1504,39.2
Contrato,2420,2370,1026,43.3
Atendimento,171,160,82,51.3
Streaming,87,87,13,14.9
Seguranca,1,1,1,100.0


In [0]:
%sql
-- ============================================================
-- Sentimento por citação (history)
-- ============================================================
-- O que faz: classifica o sentimento de cada citação (frase x aspecto) como Positivo, Negativo, Misto ou Sem avaliacao.
-- Origem:    feedback_aspectos_history
-- Saída:     tabela feedback_sentimento_history (customerID, Churn, id_frase, frase, Aspecto, qtd_pos, qtd_neg, sentimento_mencao).
-- Regras:    a análise é feita por frase inteira (do início até o ponto final), sem dividir em fragmentos por vírgula ou palavras como "but" e "however". Por isso, a frase vale para todos os aspectos citados nela, e um aspecto pode herdar o sentimento de outro. Ordem de tratamento: 1) elogio negado vira queixa ("not convenient"); 2) queixa negada vira elogio ("no issues with downtime"); 3) contagem de palavras positivas e negativas do que sobra. Positivo e negativo na mesma frase = Misto; nenhum dos dois = Sem avaliacao. Em Saida, Negativo é a frase que informa saída ou cancelamento, e Positivo é a que declara que o cliente fica ("no plans to churn"). "Switched to this provider" (entrada, não saída) pode ser marcado como Saida Negativo. As regras cobrem os padrões de texto vistos até agora.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_sentimento_history AS
WITH b1 AS (
  SELECT
    customerID, Churn, id_frase, frase, Aspecto,
    regexp_count(frase, r'(?i)\b(not|never|no longer|(was|were|is|are|did|do|does|could|would)n.t)\s+(\w+\s+){0,2}(as\s+)?(convenient|reliable|fast|consistent|reasonable|satisfied|happy|good|affordable|ideal|worth|recommend\w*|meet|resolved)\b(\s+(or|and)\s+(convenient|reliable|fast|consistent|reasonable|satisfied))?') AS n_negpos,
    regexp_replace(frase, r'(?i)\b(not|never|no longer|(was|were|is|are|did|do|does|could|would)n.t)\s+(\w+\s+){0,2}(as\s+)?(convenient|reliable|fast|consistent|reasonable|satisfied|happy|good|affordable|ideal|worth|recommend\w*|meet|resolved)\b(\s+(or|and)\s+(convenient|reliable|fast|consistent|reasonable|satisfied))?', ' ') AS s1
  FROM hackaton_databricks.tabelas_hackaton.feedback_aspectos_history
),
b2 AS (
  SELECT *,
    regexp_count(s1, r'(?i)\b(no|without( any)?|never (experienced|had)( any)?|not (experienced|had)( any)?|haven.t (had|experienced)( any)?)\s+(\w+\s+)?(issues?|problems?|complaints?|interruptions?|outages?|downtime|disruptions?|hassle|delays?|disappointments?)\b(\s+(with|from)\s+\w+(\s+\w+)?(\s+or\s+\w+(\s+\w+)?)?)?') AS n_negneg,
    regexp_replace(s1, r'(?i)\b(no|without( any)?|never (experienced|had)( any)?|not (experienced|had)( any)?|haven.t (had|experienced)( any)?)\s+(\w+\s+)?(issues?|problems?|complaints?|interruptions?|outages?|downtime|disruptions?|hassle|delays?|disappointments?)\b(\s+(with|from)\s+\w+(\s+\w+)?(\s+or\s+\w+(\s+\w+)?)?)?', ' ') AS s2
  FROM b1
),
b3 AS (
  SELECT *,
    regexp_count(s2, r'(?i)\b(reasonable|reliable|reliably|fast|convenient|conveniently|satisfied|satisfactory|happy|great|good|affordable|appreciate\w*|pleased|excellent|seamless|hassle-free|responsive|helpful|worth|recommend\w*|positive|easy|consistent|consistently|top-notch|impressed|enjoyed|smooth|stable|fair|works? well|flexib\w*|accommodat\w*)\b') + n_negneg AS qtd_pos,
    regexp_count(s2, r'(?i)\b(high|higher|expensive|steep|slow|slower|unreliable|inconvenient|inconvenience|outdated|disappointed|disappointing|disappointment|issues?|problems?|frustrat\w*|cumbersome|hassle|lacking|lack|poor|unexpected|outages?|interruptions?|disruptions?|inconsistent|unresolved|dissatisfied|unsatisfied|drawback|too|difficult|delays?|unhappy|better deal|better pricing|better options|preferred more|competitive pricing)\b') + n_negpos AS qtd_neg,
    frase rlike r'(?i)^(?!.*\b(decided|ultimately|recently|unfortunately)\b).*((\b(no|not|never|without|\w+n\Wt)\s+(\w+\s+){0,4}(plans?|intentions?|intend\w*|reasons?|desire|need|wish|considered|issues?|problems?|complaints?)\b.{0,90}\b(churn\w*|cancel\w*|switch\w*|leav\w+))|(\b(no|not|never|without|\w+n\Wt)\s+(\w+\s+){0,6}churn\w*)|((?=.*\bmonth-to-month\b).*\b(flexib\w*|freedom|option|allows? me|gives? me)\b.{0,100}\b(switch\w*|cancel\w*|leav\w+))|(\b(happy|glad|pleased)\b.{0,40}\bswitch\w*\s+to\s+(this|their)\b))' AS fica
  FROM b2
)
SELECT
  customerID, Churn, id_frase, frase, Aspecto, qtd_pos, qtd_neg,
  CASE
    WHEN Aspecto = 'Saida' THEN CASE WHEN fica THEN 'Positivo' ELSE 'Negativo' END
    WHEN qtd_pos > 0 AND qtd_neg > 0 THEN 'Misto'
    WHEN qtd_pos > 0 THEN 'Positivo'
    WHEN qtd_neg > 0 THEN 'Negativo'
    ELSE 'Sem avaliacao'
  END AS sentimento_mencao
FROM b3;

SELECT
  Aspecto,
  sentimento_mencao,
  COUNT(*) AS mencoes,
  COUNT(DISTINCT customerID) AS clientes,
  COUNT(DISTINCT CASE WHEN Churn = 'Yes' THEN customerID END) AS clientes_churn,
  ROUND(100 * COUNT(DISTINCT CASE WHEN Churn = 'Yes' THEN customerID END) / COUNT(DISTINCT customerID), 1) AS pct_churn
FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_history
GROUP BY Aspecto, sentimento_mencao
ORDER BY Aspecto, mencoes DESC;

Aspecto,sentimento_mencao,mencoes,clientes,clientes_churn,pct_churn
Atendimento,Positivo,87,87,21,24.1
Atendimento,Misto,46,45,33,73.3
Atendimento,Negativo,28,28,28,100.0
Atendimento,Sem avaliacao,10,10,10,100.0
Cobranca,Positivo,4300,4151,680,16.4
Cobranca,Misto,1032,1005,396,39.4
Cobranca,Negativo,573,539,467,86.6
Cobranca,Sem avaliacao,323,301,209,69.4
Conexao,Positivo,5635,4199,553,13.2
Conexao,Sem avaliacao,1656,1590,1022,64.3


In [0]:
%sql
-- ============================================================
-- Validação 1: frases sem avaliação que também falam de saída (history)
-- ============================================================
-- O que faz: conta, para Conexao e Contrato, as frases sem avaliação que também citam saída, separando as que parecem entrada ("switched to", "signed up") das de saída, com clientes e % de churn.
-- Origem:    feedback_sentimento_history
-- Saída:     consulta (Aspecto, tipo, mencoes, clientes, clientes_churn, pct_churn), sem criar tabela.
-- Regras:    liga cada frase ao Aspecto Saida pelo mesmo customerID e id_frase. Tipo Entrada: frase com "switched to", "signed up" ou "recently joined" e sem "cancel" ou "churn". Os demais casos são Saida.
-- ============================================================

WITH Saida AS (
  SELECT DISTINCT customerID, id_frase
  FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_history
  WHERE Aspecto = 'Saida'
)
SELECT
  s.Aspecto,
  CASE
    WHEN s.frase rlike r'(?i)(switched to|signed up|recently joined)'
     AND NOT s.frase rlike r'(?i)(cancel|churn)' THEN 'Entrada'
    ELSE 'Saida'
  END AS tipo,
  COUNT(*) AS mencoes,
  COUNT(DISTINCT s.customerID) AS clientes,
  COUNT(DISTINCT CASE WHEN s.Churn = 'Yes' THEN s.customerID END) AS clientes_churn,
  ROUND(100 * COUNT(DISTINCT CASE WHEN s.Churn = 'Yes' THEN s.customerID END) / COUNT(DISTINCT s.customerID), 1) AS pct_churn
FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_history s
JOIN Saida x ON s.customerID = x.customerID AND s.id_frase = x.id_frase
WHERE s.sentimento_mencao = 'Sem avaliacao'
  AND s.Aspecto IN ('Conexao', 'Contrato')
GROUP BY 1, 2
ORDER BY 1, 2;

Aspecto,tipo,mencoes,clientes,clientes_churn,pct_churn
Conexao,Entrada,112,112,112,100.0
Conexao,Saida,425,422,413,97.9
Contrato,Entrada,100,100,100,100.0
Contrato,Saida,225,225,206,91.6


In [0]:
%sql
-- ============================================================
-- Validação 2, parte 1: frases sem avaliação + saída, Churn No (history)
-- ============================================================
-- O que faz: lista frases Sem avaliacao de Conexao e Contrato que também citam saída, de clientes com Churn No.
-- Origem:    feedback_sentimento_history
-- Saída:     consulta (customerID, Aspecto, frase), sem criar tabela.
-- Regras:    liga cada frase ao Aspecto Saida pelo mesmo customerID e id_frase. Só Churn No. Sustenta a Validação 2 (Saida Negativo sem churn).
-- ============================================================

SELECT s.customerID, s.Aspecto, s.frase
FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_history s
JOIN (
  SELECT DISTINCT customerID, id_frase
  FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_history
  WHERE Aspecto = 'Saida'
) x ON s.customerID = x.customerID AND s.id_frase = x.id_frase
WHERE s.sentimento_mencao = 'Sem avaliacao'
  AND s.Aspecto IN ('Conexao', 'Contrato')
  AND s.Churn = 'No'
ORDER BY s.Aspecto, s.customerID;

customerID,Aspecto,frase
1047-NNCBF,Conexao,I have no plans to churn and will continue to use this internet service provider in the future.
1803-BGNBD,Conexao,I have been a customer with this internet service provider for 12 months now on a month-to-month contract with no intention of churning.
2696-RZVZW,Conexao,"While I do not have internet service with this company, I have had no reason to consider switching providers."
3528-HFRIQ,Conexao,The fact that I have been with them for 71 months without churning speaks volumes about the reliability of their internet service.
3639-XJHKQ,Conexao,I have been a customer with this internet service provider for 19 months and have been on a month-to-month contract with no plans to churn.
4667-QONEA,Conexao,"I have been a loyal customer of this DSL internet service provider for the past five years, with no intention of churning anytime soon."
5307-UVGNB,Conexao,I have been a customer with this DSL internet provider for over four years now and I have no plans to churn anytime soon.
6434-TTGJP,Conexao,I have been a customer with this DSL provider for 10 months now and I have no plans of churning anytime soon.
8646-JCOMS,Conexao,I have been a loyal customer of this fiber optic internet service provider for 66 months now and I have no intention of churning anytime soon.
0098-BOWSO,Contrato,I have been a customer with this company for 27 months on a month-to-month contract with no churn.


In [0]:
%sql
-- ============================================================
-- Validação 2: acerto texto x Churn (history)
-- ============================================================
-- O que faz: compara, por cliente, o que o texto indica (Saida Negativo, ou Negativo em Conexao, Preco, Cobranca ou Contrato) com a coluna Churn.
-- Origem:    history_tipada e feedback_sentimento_history
-- Saída:     consulta por regra (clientes, acertos, % de acerto, falsos positivos e falsos negativos), sem criar tabela.
-- Regras:    todos os clientes da base entram; quem não tem citação é classificado como Churn No. R1 = Saida Negativo. R2 = Negativo em algum dos 4 aspectos.
-- ============================================================

WITH a AS (
  SELECT
    h.customerID, h.Churn,
    COALESCE(MAX(CASE WHEN s.Aspecto = 'Saida' AND s.sentimento_mencao = 'Negativo' THEN 1 END), 0) AS r1,
    COALESCE(MAX(CASE WHEN s.Aspecto IN ('Conexao','Preco','Cobranca','Contrato')
                       AND s.sentimento_mencao = 'Negativo' THEN 1 END), 0) AS r2
  FROM hackaton_databricks.tabelas_hackaton.history_tipada h
  LEFT JOIN hackaton_databricks.tabelas_hackaton.feedback_sentimento_history s
    ON h.customerID = s.customerID
  GROUP BY h.customerID, h.Churn
),
b AS (
  SELECT 'R1 Saida Negativo' AS regra, Churn, r1 AS pred FROM a
  UNION ALL
  SELECT 'R2 4 aspectos Negativo', Churn, r2 FROM a
)
SELECT
  regra,
  COUNT(*) AS clientes,
  SUM(CASE WHEN (pred = 1 AND Churn = 'Yes') OR (pred = 0 AND Churn = 'No') THEN 1 ELSE 0 END) AS acertos,
  ROUND(100 * SUM(CASE WHEN (pred = 1 AND Churn = 'Yes') OR (pred = 0 AND Churn = 'No') THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_acerto,
  SUM(CASE WHEN pred = 1 AND Churn = 'No' THEN 1 ELSE 0 END) AS falsos_positivos,
  SUM(CASE WHEN pred = 0 AND Churn = 'Yes' THEN 1 ELSE 0 END) AS falsos_negativos
FROM b
GROUP BY regra
ORDER BY regra;

regra,clientes,acertos,pct_acerto,falsos_positivos,falsos_negativos
R1 Saida Negativo,5634,5633,100.0,1,0
R2 4 aspectos Negativo,5634,4901,87.0,168,565


In [0]:
%sql
-- ============================================================
-- Consolidação por cliente e aspecto (history)
-- ============================================================
-- O que faz: consolida as citações em uma linha por cliente e aspecto, no formato do modelo.
-- Origem:    feedback_sentimento_history
-- Saída:     tabela feedback_consolidado_history (customerID, Aspecto, Sentimento_Consolidado, Qtd_Mencoes, Qtd_Positivas, Qtd_Negativas, Qtd_Mistas, Evidencia_Mista, Evidencia_Negativa, Evidencia_Positiva).
-- Regras:    aspecto não citado não gera linha. Qtd_Mencoes inclui as citações Sem avaliacao. Qtd_Mistas é 1 se há alguma citação Misto no aspecto, senão 0, e a citação Misto não entra em Positivas nem Negativas. Sentimento_Consolidado: Misto se há Misto, ou Positivo e Negativo juntos; senão Negativo, Positivo ou Sem avaliacao. As evidências são as frases inteiras, unidas por " | ".
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_consolidado_history AS
SELECT
  customerID,
  Aspecto,
  CASE
    WHEN MAX(CASE WHEN sentimento_mencao = 'Misto' THEN 1 ELSE 0 END) = 1 THEN 'Misto'
    WHEN SUM(CASE WHEN sentimento_mencao = 'Positivo' THEN 1 ELSE 0 END) > 0
     AND SUM(CASE WHEN sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Misto'
    WHEN SUM(CASE WHEN sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Negativo'
    WHEN SUM(CASE WHEN sentimento_mencao = 'Positivo' THEN 1 ELSE 0 END) > 0 THEN 'Positivo'
    ELSE 'Sem avaliacao'
  END AS Sentimento_Consolidado,
  COUNT(*) AS Qtd_Mencoes,
  SUM(CASE WHEN sentimento_mencao = 'Positivo' THEN 1 ELSE 0 END) AS Qtd_Positivas,
  SUM(CASE WHEN sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) AS Qtd_Negativas,
  MAX(CASE WHEN sentimento_mencao = 'Misto' THEN 1 ELSE 0 END) AS Qtd_Mistas,
  array_join(collect_list(CASE WHEN sentimento_mencao = 'Misto' THEN frase END), ' | ') AS Evidencia_Mista,
  array_join(collect_list(CASE WHEN sentimento_mencao = 'Negativo' THEN frase END), ' | ') AS Evidencia_Negativa,
  array_join(collect_list(CASE WHEN sentimento_mencao = 'Positivo' THEN frase END), ' | ') AS Evidencia_Positiva
FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_history
GROUP BY customerID, Aspecto;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history
ORDER BY customerID, Aspecto
LIMIT ALL;

customerID,Aspecto,Sentimento_Consolidado,Qtd_Mencoes,Qtd_Positivas,Qtd_Negativas,Qtd_Mistas,Evidencia_Mista,Evidencia_Negativa,Evidencia_Positiva
0002-ORFBO,Cobranca,Positivo,1,1,0,0,,,I appreciate the affordable monthly charge of $65.6 and the option to pay by mailed check.
0002-ORFBO,Conexao,Positivo,1,1,0,0,,,The DSL internet service has been reliable and I have not had any disruptions in my connection.
0002-ORFBO,Preco,Positivo,1,1,0,0,,,I appreciate the affordable monthly charge of $65.6 and the option to pay by mailed check.
0002-ORFBO,Saida,Positivo,1,1,0,0,,,"Overall, I am satisfied with my experience and have no plans to churn."
0003-MKNFE,Cobranca,Positivo,1,1,0,0,,,I appreciate the option to pay by mailed check as it is convenient for me.
0003-MKNFE,Conexao,Positivo,2,2,0,0,,,The DSL internet has been reliable and I have not experienced any issues with connectivity. | I have been a customer with this internet service provider for 9 months now and I am very satisfied with the service.
0003-MKNFE,Preco,Positivo,1,1,0,0,,,The monthly charges of $59.90 are reasonable for the service provided.
0011-IGKFF,Cobranca,Misto,1,0,0,1,"Additionally, the payment method of electronic check was convenient, but I was hoping for more competitive pricing options and possibly a discount for long-term customers.",,
0011-IGKFF,Conexao,Misto,1,0,0,1,"While the fiber optic internet service was fast and reliable, I found the monthly charges of $98 to be quite high.",,
0011-IGKFF,Preco,Misto,3,1,0,1,"Additionally, the payment method of electronic check was convenient, but I was hoping for more competitive pricing options and possibly a discount for long-term customers. | While the fiber optic internet service was fast and reliable, I found the monthly charges of $98 to be quite high.",,"Overall, I had a decent experience with the service, but ultimately decided to switch to a more affordable option."


In [0]:
%sql
-- ============================================================
-- Validação 3: contagem por sentimento consolidado (history)
-- ============================================================
-- O que faz: conta as linhas (cliente x aspecto) por Sentimento_Consolidado, com o percentual do total.
-- Origem:    feedback_consolidado_history
-- Saída:     consulta (Sentimento_Consolidado, linhas, pct_total), sem criar tabela.
-- Regras:    o percentual é sobre o total de linhas da tabela.
-- ============================================================

SELECT
  Sentimento_Consolidado,
  COUNT(*) AS linhas,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_total
FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_history
GROUP BY Sentimento_Consolidado
ORDER BY linhas DESC;

Sentimento_Consolidado,linhas,pct_total
Positivo,15482,66.5
Misto,3308,14.2
Negativo,2436,10.5
Sem avaliacao,2063,8.9


In [0]:
%sql
-- ============================================================
-- Classificação por cliente (history)
-- ============================================================
-- O que faz: classifica cada cliente como Churn Yes ou No pelo texto, uma linha por customerID.
-- Origem:    history_tipada e feedback_sentimento_history
-- Saída:     tabela feedback_classificacao_history (customerID, Churn, Churn_R1, Churn_R2, Churn_R).
-- Regras:    R1 = Yes se há Saida Negativo. R2 = Yes se há Negativo em Conexao, Preco, Cobranca ou Contrato. Churn_R = Yes se R1 ou R2 for Yes. Quem não tem citação é No.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_classificacao_history AS
SELECT
  h.customerID,
  h.Churn,
  CASE WHEN MAX(CASE WHEN s.Aspecto = 'Saida' AND s.sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) = 1
       THEN 'Yes' ELSE 'No' END AS Churn_R1,
  CASE WHEN MAX(CASE WHEN s.Aspecto IN ('Conexao','Preco','Cobranca','Contrato')
                      AND s.sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) = 1
       THEN 'Yes' ELSE 'No' END AS Churn_R2,
  CASE WHEN MAX(CASE WHEN s.Aspecto IN ('Saida','Conexao','Preco','Cobranca','Contrato')
                      AND s.sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) = 1
       THEN 'Yes' ELSE 'No' END AS Churn_R
FROM hackaton_databricks.tabelas_hackaton.history_tipada h
LEFT JOIN hackaton_databricks.tabelas_hackaton.feedback_sentimento_history s
  ON h.customerID = s.customerID
GROUP BY h.customerID, h.Churn;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.feedback_classificacao_history
ORDER BY customerID
LIMIT 20;

customerID,Churn,Churn_R1,Churn_R2,Churn_R
0002-ORFBO,No,No,No,No
0003-MKNFE,No,No,No,No
0011-IGKFF,Yes,Yes,No,Yes
0013-MHZWF,No,No,No,No
0013-SMEOE,No,No,No,No
0014-BMAQU,No,No,No,No
0015-UOCOJ,No,No,No,No
0016-QLJIS,No,No,No,No
0017-DINOC,No,No,No,No
0017-IUDMW,No,No,No,No


In [0]:
%sql
-- ============================================================
-- Validação 4: concordância das regras com o Churn (history)
-- ============================================================
-- O que faz: compara Churn_R1, Churn_R2 e Churn_R com a coluna Churn e calcula acerto, precisão e cobertura de cada regra.
-- Origem:    feedback_classificacao_history
-- Saída:     consulta, uma linha por regra (vp, fp, fn, vn, acerto_pct, precisao_pct, cobertura_pct), sem criar tabela.
-- Regras:    Yes é a classe positiva. vp = texto Yes e Churn Yes; fp = texto Yes e Churn No; fn = texto No e Churn Yes; vn = texto No e Churn No. Acerto = (vp + vn) / total de clientes. Precisão = vp / (vp + fp). Cobertura = vp / (vp + fn). Cliente com texto de saída e Churn No (ex.: recontratou) conta como fp.
-- ============================================================

SELECT 'R1' AS regra,
  SUM(CASE WHEN Churn_R1 = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END) AS vp,
  SUM(CASE WHEN Churn_R1 = 'Yes' AND Churn = 'No'  THEN 1 ELSE 0 END) AS fp,
  SUM(CASE WHEN Churn_R1 = 'No'  AND Churn = 'Yes' THEN 1 ELSE 0 END) AS fn,
  SUM(CASE WHEN Churn_R1 = 'No'  AND Churn = 'No'  THEN 1 ELSE 0 END) AS vn,
  ROUND(100 * SUM(CASE WHEN Churn_R1 = Churn THEN 1 ELSE 0 END) / COUNT(*), 1) AS acerto_pct,
  ROUND(100 * SUM(CASE WHEN Churn_R1 = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN Churn_R1 = 'Yes' THEN 1 ELSE 0 END), 0), 1) AS precisao_pct,
  ROUND(100 * SUM(CASE WHEN Churn_R1 = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END), 0), 1) AS cobertura_pct
FROM hackaton_databricks.tabelas_hackaton.feedback_classificacao_history
UNION ALL
SELECT 'R2',
  SUM(CASE WHEN Churn_R2 = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN Churn_R2 = 'Yes' AND Churn = 'No'  THEN 1 ELSE 0 END),
  SUM(CASE WHEN Churn_R2 = 'No'  AND Churn = 'Yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN Churn_R2 = 'No'  AND Churn = 'No'  THEN 1 ELSE 0 END),
  ROUND(100 * SUM(CASE WHEN Churn_R2 = Churn THEN 1 ELSE 0 END) / COUNT(*), 1),
  ROUND(100 * SUM(CASE WHEN Churn_R2 = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN Churn_R2 = 'Yes' THEN 1 ELSE 0 END), 0), 1),
  ROUND(100 * SUM(CASE WHEN Churn_R2 = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END), 0), 1)
FROM hackaton_databricks.tabelas_hackaton.feedback_classificacao_history
UNION ALL
SELECT 'R',
  SUM(CASE WHEN Churn_R = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN Churn_R = 'Yes' AND Churn = 'No'  THEN 1 ELSE 0 END),
  SUM(CASE WHEN Churn_R = 'No'  AND Churn = 'Yes' THEN 1 ELSE 0 END),
  SUM(CASE WHEN Churn_R = 'No'  AND Churn = 'No'  THEN 1 ELSE 0 END),
  ROUND(100 * SUM(CASE WHEN Churn_R = Churn THEN 1 ELSE 0 END) / COUNT(*), 1),
  ROUND(100 * SUM(CASE WHEN Churn_R = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN Churn_R = 'Yes' THEN 1 ELSE 0 END), 0), 1),
  ROUND(100 * SUM(CASE WHEN Churn_R = 'Yes' AND Churn = 'Yes' THEN 1 ELSE 0 END)
        / NULLIF(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END), 0), 1)
FROM hackaton_databricks.tabelas_hackaton.feedback_classificacao_history;

regra,vp,fp,fn,vn,acerto_pct,precisao_pct,cobertura_pct
R1,1504,1,0,4129,100.0,99.9,100.0
R2,939,168,565,3962,87.0,84.8,62.4
R,1504,169,0,3961,97.0,89.9,100.0


In [0]:
%sql
-- ============================================================
-- Frases do feedback (inference)
-- ============================================================
-- O que faz: divide o CustomerFeedback em frases, uma linha por frase, mantendo customerID. É a base das células de aspectos e sentimento.
-- Origem:    inference_tipada
-- Saída:     tabela feedback_frases_inference (customerID, id_frase, frase).
-- Regras:    a frase termina em ".", "!" ou "?" seguido de espaço ou quebra de linha, então valores como 73.9 não são cortados. Frases vazias e feedbacks nulos são descartados. id_frase começa em 0. A regex usa "\\s" (barra dupla), exigida em SQL no Databricks.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_frases_inference AS
SELECT
  h.customerID,
  p.pos AS id_frase,
  trim(p.col) AS frase
FROM hackaton_databricks.tabelas_hackaton.inference_tipada h,
     LATERAL posexplode(split(h.CustomerFeedback, '(?<=[.!?])\\s+')) AS p
WHERE h.CustomerFeedback IS NOT NULL
  AND trim(p.col) <> '';

SELECT
  COUNT(DISTINCT customerID) AS clientes,
  COUNT(*) AS frases,
  ROUND(COUNT(*) / COUNT(DISTINCT customerID), 1) AS frases_por_cliente
FROM hackaton_databricks.tabelas_hackaton.feedback_frases_inference;

clientes,frases,frases_por_cliente
1409,6369,4.5


In [0]:
%sql
-- ============================================================
-- Citações por frase e aspecto (inference)
-- ============================================================
-- O que faz: detecta, por frase, quais aspectos são citados, usando um dicionário de termos por aspecto. Só marca a menção; o sentimento vem na célula seguinte.
-- Origem:    feedback_frases_inference
-- Saída:     tabela feedback_aspectos_inference (customerID, id_frase, frase, Aspecto), uma linha por frase e aspecto.
-- Regras:    o dicionário é em regex, sem distinção de maiúsculas e minúsculas, e uma frase pode gerar mais de um aspecto. Aspectos: Conexao, Preco, Cobranca, Contrato, Telefone, Seguranca, Backup, Protecao, Atendimento, Streaming e Saida. Os termos vêm da leitura de 100 textos e da lista de serviços. Preco inclui símbolo de moeda seguido de número, escrito como código hexadecimal x24 para o Databricks não lê-lo como parâmetro. Streaming inclui "stream", "movie" e "tv", e parte das menções descreve uso da internet, não o serviço contratado. Saida marca só a menção (churn, cancel, switch, discontinu, leav), sem tratar negação; a negação é tratada na célula de sentimento.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_aspectos_inference AS
WITH dicionario AS (
  SELECT * FROM VALUES
    ('Conexao',   r'(?i)\b(internet|dsl|fiber|speeds?|connection|connectivity|outages?|downtime|disruptions?|interruptions?)\b'),
    ('Preco',     r'(?i)(\x24\s?\d|\b(charges?|prices?|pricing|expensive|afford\w*|costs?|steep|deals?|cheap\w*|fees?)\b)'),
    ('Cobranca',  r'(?i)\b(pay|pays|paying|paid|payments?|checks?|bank transfers?|credit card|bill|bills|billing|paperless)\b'),
    ('Contrato',  r'(?i)\b(contracts?|month-to-month|one-year|two-year|one year|two year)\b'),
    ('Telefone',  r'(?i)\b(phone|multiple lines?)\b'),
    ('Seguranca', r'(?i)\bsecurity\b'),
    ('Backup',    r'(?i)\bback-?up\b'),
    ('Protecao',  r'(?i)\bprotection\b'),
    ('Atendimento',   r'(?i)\b(tech support|technical support|customer service|customer support|support)\b'),
    ('Streaming', r'(?i)\b(stream\w*|movies?|tv)\b'),
    ('Saida',     r'(?i)\b(churn\w*|cancel\w*|switch\w*|discontinu\w*|leav\w+)\b')
  AS d(Aspecto, regex)
)
SELECT
  f.customerID,
  f.id_frase,
  f.frase,
  d.Aspecto
FROM hackaton_databricks.tabelas_hackaton.feedback_frases_inference f
JOIN dicionario d
  ON f.frase rlike d.regex;

SELECT
  Aspecto,
  COUNT(*) AS frases,
  COUNT(DISTINCT customerID) AS clientes
  FROM hackaton_databricks.tabelas_hackaton.feedback_aspectos_inference
GROUP BY Aspecto
ORDER BY frases DESC;

Aspecto,frases,clientes
Conexao,2241,1407
Preco,1724,1400
Cobranca,1545,1393
Saida,1297,986
Contrato,629,610
Atendimento,36,32
Streaming,24,24


In [0]:
%sql
-- ============================================================
-- Sentimento por citação (inference)
-- ============================================================
-- O que faz: classifica o sentimento de cada citação (frase x aspecto) como Positivo, Negativo, Misto ou Sem avaliacao.
-- Origem:    feedback_aspectos_inference
-- Saída:     tabela feedback_sentimento_inference (customerID, id_frase, frase, Aspecto, qtd_pos, qtd_neg, sentimento_mencao).
-- Regras:    a análise é feita por frase inteira (do início até o ponto final), sem dividir em fragmentos por vírgula ou palavras como "but" e "however". Por isso, a frase vale para todos os aspectos citados nela, e um aspecto pode herdar o sentimento de outro. Ordem de tratamento: 1) elogio negado vira queixa ("not convenient"); 2) queixa negada vira elogio ("no issues with downtime"); 3) contagem de palavras positivas e negativas do que sobra. Positivo e negativo na mesma frase = Misto; nenhum dos dois = Sem avaliacao. Em Saida, Negativo é a frase que informa saída ou cancelamento, e Positivo é a que declara que o cliente fica ("no plans to churn"). "Switched to this provider" (entrada, não saída) pode ser marcado como Saida Negativo. As regras cobrem os padrões de texto vistos até agora.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_sentimento_inference AS
WITH b1 AS (
  SELECT
    customerID, id_frase, frase, Aspecto,
    regexp_count(frase, r'(?i)\b(not|never|no longer|(was|were|is|are|did|do|does|could|would)n.t)\s+(\w+\s+){0,2}(as\s+)?(convenient|reliable|fast|consistent|reasonable|satisfied|happy|good|affordable|ideal|worth|recommend\w*|meet|resolved)\b(\s+(or|and)\s+(convenient|reliable|fast|consistent|reasonable|satisfied))?') AS n_negpos,
    regexp_replace(frase, r'(?i)\b(not|never|no longer|(was|were|is|are|did|do|does|could|would)n.t)\s+(\w+\s+){0,2}(as\s+)?(convenient|reliable|fast|consistent|reasonable|satisfied|happy|good|affordable|ideal|worth|recommend\w*|meet|resolved)\b(\s+(or|and)\s+(convenient|reliable|fast|consistent|reasonable|satisfied))?', ' ') AS s1
  FROM hackaton_databricks.tabelas_hackaton.feedback_aspectos_inference
),
b2 AS (
  SELECT *,
    regexp_count(s1, r'(?i)\b(no|without( any)?|never (experienced|had)( any)?|not (experienced|had)( any)?|haven.t (had|experienced)( any)?)\s+(\w+\s+)?(issues?|problems?|complaints?|interruptions?|outages?|downtime|disruptions?|hassle|delays?|disappointments?)\b(\s+(with|from)\s+\w+(\s+\w+)?(\s+or\s+\w+(\s+\w+)?)?)?') AS n_negneg,
    regexp_replace(s1, r'(?i)\b(no|without( any)?|never (experienced|had)( any)?|not (experienced|had)( any)?|haven.t (had|experienced)( any)?)\s+(\w+\s+)?(issues?|problems?|complaints?|interruptions?|outages?|downtime|disruptions?|hassle|delays?|disappointments?)\b(\s+(with|from)\s+\w+(\s+\w+)?(\s+or\s+\w+(\s+\w+)?)?)?', ' ') AS s2
  FROM b1
),
b3 AS (
  SELECT *,
    regexp_count(s2, r'(?i)\b(reasonable|reliable|reliably|fast|convenient|conveniently|satisfied|satisfactory|happy|great|good|affordable|appreciate\w*|pleased|excellent|seamless|hassle-free|responsive|helpful|worth|recommend\w*|positive|easy|consistent|consistently|top-notch|impressed|enjoyed|smooth|stable|fair|works? well|flexib\w*|accommodat\w*)\b') + n_negneg AS qtd_pos,
    regexp_count(s2, r'(?i)\b(high|higher|expensive|steep|slow|slower|unreliable|inconvenient|inconvenience|outdated|disappointed|disappointing|disappointment|issues?|problems?|complaints?|frustrat\w*|cumbersome|hassle|lacking|lack|poor|unexpected|outages?|interruptions?|disruptions?|inconsistent|unresolved|dissatisfied|unsatisfied|drawback|too|difficult|delays?|unhappy|better deal|better pricing|better options|preferred more|competitive pricing)\b') + n_negpos AS qtd_neg,
    frase rlike r'(?i)^(?!.*\b(decided|ultimately|recently|unfortunately)\b).*((\b(no|not|never|without|\w+n\Wt)\s+(\w+\s+){0,4}(plans?|intentions?|intend\w*|reasons?|desire|need|wish|considered|issues?|problems?|complaints?)\b.{0,90}\b(churn\w*|cancel\w*|switch\w*|leav\w+))|(\b(no|not|never|without|\w+n\Wt)\s+(\w+\s+){0,6}churn\w*)|((?=.*\bmonth-to-month\b).*\b(flexib\w*|freedom|option|allows? me|gives? me)\b.{0,100}\b(switch\w*|cancel\w*|leav\w+))|(\b(happy|glad|pleased)\b.{0,40}\bswitch\w*\s+to\s+(this|their)\b))' AS fica
  FROM b2
)
SELECT
  customerID, id_frase, frase, Aspecto, qtd_pos, qtd_neg,
  CASE
    WHEN Aspecto = 'Saida' THEN CASE WHEN fica THEN 'Positivo' ELSE 'Negativo' END
    WHEN qtd_pos > 0 AND qtd_neg > 0 THEN 'Misto'
    WHEN qtd_pos > 0 THEN 'Positivo'
    WHEN qtd_neg > 0 THEN 'Negativo'
    ELSE 'Sem avaliacao'
  END AS sentimento_mencao
FROM b3;

SELECT
  Aspecto,
  sentimento_mencao,
  COUNT(*) AS mencoes,
  COUNT(DISTINCT customerID) AS clientes
FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_inference
GROUP BY Aspecto, sentimento_mencao
ORDER BY Aspecto, mencoes DESC;

Aspecto,sentimento_mencao,mencoes,clientes
Atendimento,Positivo,21,21
Atendimento,Misto,7,7
Atendimento,Negativo,5,5
Atendimento,Sem avaliacao,3,3
Cobranca,Positivo,1049,1012
Cobranca,Misto,268,260
Cobranca,Negativo,160,154
Cobranca,Sem avaliacao,68,68
Conexao,Positivo,1424,1059
Conexao,Sem avaliacao,440,427


In [0]:
gua%sql
-- ============================================================
-- Consolidação por cliente e aspecto (inference)
-- ============================================================
-- O que faz: consolida as citações em uma linha por cliente e aspecto, no formato do modelo (André), para comparar com a Terra Signal.
-- Origem:    feedback_sentimento_inference
-- Saída:     tabela feedback_consolidado_inference (customerID, Aspecto, Sentimento_Consolidado, Qtd_Mencoes, Qtd_Positivas, Qtd_Negativas, Qtd_Mistas, Evidencia_Mista, Evidencia_Negativa, Evidencia_Positiva).
-- Regras:    aspecto não citado não gera linha. Qtd_Mencoes inclui as citações Sem avaliacao. Qtd_Mistas é 1 se há alguma citação Misto no aspecto, senão 0, e a citação Misto não entra em Positivas nem Negativas. Sentimento_Consolidado: Misto se há Misto, ou Positivo e Negativo juntos; senão Negativo, Positivo ou Sem avaliacao. As evidências são as frases inteiras, unidas por " | ".
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.feedback_consolidado_inference AS
SELECT
  customerID,
  Aspecto,
  CASE
    WHEN MAX(CASE WHEN sentimento_mencao = 'Misto' THEN 1 ELSE 0 END) = 1 THEN 'Misto'
    WHEN SUM(CASE WHEN sentimento_mencao = 'Positivo' THEN 1 ELSE 0 END) > 0
     AND SUM(CASE WHEN sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Misto'
    WHEN SUM(CASE WHEN sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) > 0 THEN 'Negativo'
    WHEN SUM(CASE WHEN sentimento_mencao = 'Positivo' THEN 1 ELSE 0 END) > 0 THEN 'Positivo'
    ELSE 'Sem avaliacao'
  END AS Sentimento_Consolidado,
  COUNT(*) AS Qtd_Mencoes,
  SUM(CASE WHEN sentimento_mencao = 'Positivo' THEN 1 ELSE 0 END) AS Qtd_Positivas,
  SUM(CASE WHEN sentimento_mencao = 'Negativo' THEN 1 ELSE 0 END) AS Qtd_Negativas,
  MAX(CASE WHEN sentimento_mencao = 'Misto' THEN 1 ELSE 0 END) AS Qtd_Mistas,
  array_join(collect_list(CASE WHEN sentimento_mencao = 'Misto' THEN frase END), ' | ') AS Evidencia_Mista,
  array_join(collect_list(CASE WHEN sentimento_mencao = 'Negativo' THEN frase END), ' | ') AS Evidencia_Negativa,
  array_join(collect_list(CASE WHEN sentimento_mencao = 'Positivo' THEN frase END), ' | ') AS Evidencia_Positiva
FROM hackaton_databricks.tabelas_hackaton.feedback_sentimento_inference
GROUP BY customerID, Aspecto;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_inference
ORDER BY customerID, Aspecto
LIMIT ALL;

customerID,Aspecto,Sentimento_Consolidado,Qtd_Mencoes,Qtd_Positivas,Qtd_Negativas,Qtd_Mistas,Evidencia_Mista,Evidencia_Negativa,Evidencia_Positiva
0004-TLHLJ,Cobranca,Negativo,1,0,1,0,,"Additionally, the payment method of electronic check was not convenient for me.",
0004-TLHLJ,Conexao,Negativo,2,0,1,0,,I found that the internet speeds were not consistent and frequently experienced interruptions in my service.,
0004-TLHLJ,Contrato,Sem avaliacao,1,0,0,0,,,
0004-TLHLJ,Preco,Sem avaliacao,1,0,0,0,,,
0004-TLHLJ,Saida,Negativo,2,0,2,0,,"However, I have decided to churn and discontinue my service. | I recently switched to this fiber optic internet service provider four months ago, opting for a month-to-month contract with a monthly charge of $73.9.",
0013-EXCHZ,Cobranca,Misto,1,0,0,1,"The monthly charges of $83.9 for fiber optic internet were reasonable, but I found the process of sending a mailed check for payment to be inconvenient.",,
0013-EXCHZ,Conexao,Misto,1,0,0,1,"The monthly charges of $83.9 for fiber optic internet were reasonable, but I found the process of sending a mailed check for payment to be inconvenient.",,
0013-EXCHZ,Preco,Misto,1,0,0,1,"The monthly charges of $83.9 for fiber optic internet were reasonable, but I found the process of sending a mailed check for payment to be inconvenient.",,
0013-EXCHZ,Saida,Negativo,2,0,2,0,,"I recently canceled my service after only 3 months with this company. | Overall, I was not satisfied with the level of service provided and decided to switch to a different provider.",
0020-JDNXP,Cobranca,Positivo,1,1,0,0,,,I appreciate being able to pay via mailed check as it is convenient for me.


In [0]:
%sql
-- ============================================================
-- Validação 5: contagem por sentimento consolidado (inference)
-- ============================================================
-- O que faz: conta as linhas (cliente x aspecto) por Sentimento_Consolidado, com o percentual do total.
-- Origem:    feedback_consolidado_inference
-- Saída:     consulta (Sentimento_Consolidado, linhas, pct_total), sem criar tabela.
-- Regras:    o percentual é sobre o total de linhas da tabela.
-- ============================================================

SELECT
  Sentimento_Consolidado,
  COUNT(*) AS linhas,
  ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_total
FROM hackaton_databricks.tabelas_hackaton.feedback_consolidado_inference
GROUP BY Sentimento_Consolidado
ORDER BY linhas DESC;

Sentimento_Consolidado,linhas,pct_total
Positivo,3855,65.9
Misto,872,14.9
Negativo,603,10.3
Sem avaliacao,522,8.9


In [0]:
%sql
-- ============================================================
-- Tabela terra_signal_history_aspects (Terra Signal, history)
-- ============================================================
-- O que faz: carrega o arquivo CSV da Terra Signal (aspectos, history) em tabela.
-- Origem:    terra_signal_history_aspects.csv (volume bases, schema bases)
-- Saída:     tabela terra_signal_history_aspects, com as colunas do CSV, todas como texto.
-- Regras:    separador do CSV ";" e cabeçalho na primeira linha. Os tipos não são inferidos.
-- ============================================================

CREATE OR REPLACE TABLE hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects AS
SELECT *
FROM read_files(
  '/Volumes/hackaton_databricks/bases/bases/terra_signal_history_aspects.csv',
  format => 'csv',
  sep => ';',
  header => true,
  inferColumnTypes => false
);

SELECT COUNT(*) AS linhas
FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects;

SELECT *
FROM hackaton_databricks.tabelas_hackaton.terra_signal_history_aspects
LIMIT ALL;

customerID,Aspecto,Sentimento_Consolidado,Qtd_Mencoes,Qtd_Positivas,Qtd_Negativas,Qtd_Mistas,Evidencia_Mista,Evidencia_Negativa,Evidencia_Positiva,_rescued_data
0002-ORFBO,Cobranca,Sem avaliacao,1,0,0,0,null,null,null,null
0002-ORFBO,Conexao,Positivo,1,1,0,0,null,null,The DSL internet service has been reliable and I have not had any disruptions in my connection.,null
0002-ORFBO,Preco,Positivo,1,1,0,0,null,null,I appreciate the affordable monthly charge of $65.6 and the option to pay by mailed check.,null
0003-MKNFE,Cobranca,Positivo,1,1,0,0,null,null,I appreciate the option to pay by mailed check as it is convenient for me.,null
0003-MKNFE,Conexao,Positivo,2,1,0,0,null,null,The DSL internet has been reliable and I have not experienced any issues with connectivity.,null
0003-MKNFE,Preco,Positivo,1,1,0,0,null,null,The monthly charges of $59.90 are reasonable for the service provided.,null
0011-IGKFF,Cobranca,Positivo,1,1,0,0,null,null,the payment method of electronic check was convenient,null
0011-IGKFF,Conexao,Positivo,1,1,0,0,null,null,While the fiber optic internet service was fast and reliable,null
0011-IGKFF,Preco,Positivo,3,2,0,0,null,null,"but I was hoping for more competitive pricing options and possibly a discount for long-term customers. | but ultimately decided to switch to a more affordable option.""""",null
0013-MHZWF,Cobranca,Sem avaliacao,1,0,0,0,null,null,null,null
